In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Load and merge data
gen_df = pd.read_csv("../data/processed/karnataka_generation_data.csv")
weather_df = pd.read_csv("../data/synthetic_weather_data.csv")

# Merge on timestamp
df = pd.merge(gen_df, weather_df, on='timestamp', how='inner')

# Basic preprocessing
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['month'] = df['timestamp'].dt.month

# Select one plant for simplicity (Pavagada Solar)
plant_df = df[df['plant_name'] == 'Pavagada_Solar_Park_Phase1'].copy()

# Features and target
feature_cols = ['hour', 'month', 'temperature_2m', 'wind_speed',
                'surface_solar_radiation_downwards', 'total_cloud_cover', 'capacity_mw']
X = plant_df[feature_cols]
y = plant_df['generation_mw']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LGBMRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluate
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"MAE: {mae:.2f} MW")
print(f"RMSE: {rmse:.2f} MW")

# Plot results
plt.figure(figsize=(12, 6))
plt.plot(y_test.values[:200], label="Actual", alpha=0.7)
plt.plot(y_pred[:200], label="Predicted", alpha=0.7)
plt.legend()
plt.title("Solar Generation Forecast - Pavagada Solar Park")
plt.xlabel("Time Steps")
plt.ylabel("Generation (MW)")
plt.show()

print("Basic forecasting model completed successfully!")